# 02 — Fine-tune VideoMAEv2-Small for VSL Recognition

**Model**: VideoMAEv2-Small (distilled, ~22M params)  
**Dataset**: Multi-VSL (WACV 2025) — 50 classes, frontal view  
**Training**: Google Colab / Kaggle T4 16GB  
**Inference target**: Local GPU 4GB VRAM

## Architecture
```
Video (16 frames × 224×224) → VideoMAEv2-Small Encoder (ViT-S) → [CLS] token → FC (50 classes) → Softmax
```

**Pre-trained on**: Kinetics-400 (video action recognition)  
**Fine-tune on**: Multi-VSL Vietnamese Sign Language

In [ ]:
# ============================================================
# CẤU HÌNH TRAINING
# ============================================================

# Model
MODEL_NAME = "MCG-NJU/videomae-small-finetuned-kinetics"  # Pre-trained VideoMAE-Small
NUM_CLASSES = 50
NUM_FRAMES = 16
IMAGE_SIZE = 224

# Training hyperparameters
BATCH_SIZE = 8            # 8 works well on T4 16GB with fp16
LEARNING_RATE = 5e-4      # Fine-tuning LR
WEIGHT_DECAY = 0.05       # AdamW weight decay
EPOCHS = 30               # 30-50 epochs
WARMUP_EPOCHS = 5         # Linear warmup
LABEL_SMOOTHING = 0.1     # Label smoothing for regularization
FP16 = True               # Mixed precision (saves VRAM)

# Data
VAL_RATIO = 0.2
NUM_WORKERS = 2           # DataLoader workers
SEED = 42

## 1. Setup

In [ ]:
import os
import sys

# === Detect environment ===
def detect_environment():
    try:
        import google.colab
        return "colab"
    except ImportError:
        pass
    if os.path.exists("/kaggle/working"):
        return "kaggle"
    return "local"

ENV = detect_environment()
print(f"🖥️ Environment: {ENV}")

# === Paths ===
if ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/vsl-recognition"
elif ENV == "kaggle":
    BASE_DIR = "/kaggle/working/vsl-recognition"
else:
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA_DIR = os.path.join(BASE_DIR, "data", "multi_vsl")
MODEL_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"📁 Base: {BASE_DIR}")
print(f"📁 Data: {DATA_DIR}")
print(f"📁 Models: {MODEL_DIR}")

In [ ]:
# === Install dependencies ===
if ENV in ("colab", "kaggle"):
    !pip install -q transformers accelerate decord gdown

import json
import time
import random
import numpy as np
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torchvision.transforms import (
    Resize, CenterCrop, RandomResizedCrop, ColorJitter, Normalize
)
from torchvision.transforms.functional import (
    adjust_brightness, adjust_contrast, adjust_saturation
)


## 2. Dataset & DataLoader

In [ ]:
# === Video loading ===
def load_video(video_path: str, num_frames: int = 16) -> np.ndarray:
    """Load video and uniformly sample frames.
    Returns: np.ndarray (num_frames, H, W, 3) uint8.
    """
    try:
        from decord import VideoReader, cpu
        vr = VideoReader(video_path, ctx=cpu(0))
        total = len(vr)
        if total >= num_frames:
            indices = np.linspace(0, total - 1, num_frames, dtype=int)
        else:
            indices = np.arange(total)
            indices = np.concatenate([indices, np.full(num_frames - total, total - 1, dtype=int)])
        return vr.get_batch(indices).asnumpy()
    except Exception:
        import cv2
        cap = cv2.VideoCapture(video_path)
        frames = []
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        cap.release()
        if not frames:
            raise ValueError(f"Cannot read: {video_path}")
        frames = np.array(frames)
        total = len(frames)
        if total >= num_frames:
            indices = np.linspace(0, total - 1, num_frames, dtype=int)
        else:
            indices = np.concatenate([np.arange(total), np.full(num_frames - total, total - 1, dtype=int)])
        return frames[indices]


class VSLVideoDataset(Dataset):
    """Video dataset for Multi-VSL."""
    
    def __init__(self, video_list: list, num_frames: int = 16, 
                 image_size: int = 224, mode: str = "train"):
        self.video_list = video_list  # list of {"path": ..., "label": ...}
        self.num_frames = num_frames
        self.image_size = image_size
        self.mode = mode
        self.normalize = Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    
    def __len__(self):
        return len(self.video_list)
    
    def __getitem__(self, idx):
        item = self.video_list[idx]
        video_path = item["path"]
        label = item["label"]
        
        # Load frames
        frames = load_video(video_path, self.num_frames)  # (T, H, W, 3)
        
        # Convert to tensor
        video = torch.from_numpy(frames).float() / 255.0
        video = video.permute(0, 3, 1, 2)  # (T, 3, H, W)
        
        # Apply transforms
        T = video.shape[0]
        transformed = []
        
        if self.mode == "train":
            # Random crop (same params for all frames)
            i, j, h, w = RandomResizedCrop.get_params(
                video[0], scale=(0.8, 1.0), ratio=(0.9, 1.1)
            )
            # Sample color jitter params once for temporal consistency
            _, brightness_factor, contrast_factor, saturation_factor, _ = \
                ColorJitter.get_params(
                    brightness=(0.9, 1.1),
                    contrast=(0.9, 1.1),
                    saturation=(0.9, 1.1),
                    hue=None,
                )
            
            for t in range(T):
                frame = video[t][:, i:i+h, j:j+w]
                frame = Resize((self.image_size, self.image_size), antialias=True)(frame)
                frame = adjust_brightness(frame, brightness_factor)
                frame = adjust_contrast(frame, contrast_factor)
                frame = adjust_saturation(frame, saturation_factor)
                frame = self.normalize(frame)
                transformed.append(frame)
        else:
            for t in range(T):
                frame = Resize(self.image_size + 32, antialias=True)(video[t])
                frame = CenterCrop(self.image_size)(frame)
                frame = self.normalize(frame)
                transformed.append(frame)
        
        video_tensor = torch.stack(transformed)  # (T, 3, H, W)
        return video_tensor, label

print("✅ Dataset class defined")

In [ ]:
# === Load split from notebook 01 ===
meta_dir = Path(DATA_DIR).parent

# Check if split exists
if (meta_dir / "train.json").exists():
    with open(meta_dir / "train.json") as f:
        train_data = json.load(f)
    with open(meta_dir / "val.json") as f:
        val_data = json.load(f)
    with open(meta_dir / "metadata.json") as f:
        metadata = json.load(f)
    
    class_names = metadata["class_names"]
    print(f"✅ Loaded split from notebook 01:")
else:
    # Create split on-the-fly if not found
    print("⚠️ Split not found, creating from data directory...")
    data_path = Path(DATA_DIR)
    all_classes = sorted([d for d in data_path.iterdir() if d.is_dir()])[:NUM_CLASSES]
    class_names = [d.name for d in all_classes]
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    
    train_data, val_data = [], []
    for cls_dir in all_classes:
        videos = sorted([str(f) for f in cls_dir.iterdir() if f.suffix.lower() in ('.avi', '.mp4')])
        random.shuffle(videos)
        split = max(1, int(len(videos) * 0.8))
        for v in videos[:split]:
            train_data.append({"path": v, "label": class_to_idx[cls_dir.name]})
        for v in videos[split:]:
            val_data.append({"path": v, "label": class_to_idx[cls_dir.name]})

print(f"   Classes: {len(class_names)}")
print(f"   Train: {len(train_data)} videos")
print(f"   Val: {len(val_data)} videos")

# Create DataLoaders
train_dataset = VSLVideoDataset(train_data, NUM_FRAMES, IMAGE_SIZE, mode="train")
val_dataset = VSLVideoDataset(val_data, NUM_FRAMES, IMAGE_SIZE, mode="eval")

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f"\
   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")

## 3. Load VideoMAEv2-Small Pre-trained Model

In [ ]:
# === Load pre-trained VideoMAE model ===
print(f"📥 Loading pre-trained model: {MODEL_NAME}")
print(f"   (This downloads ~90MB on first run)\
")

model = VideoMAEForVideoClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,  # Replace classification head
)
model = model.to(device)

# Model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model loaded!")
print(f"   Total params: {total_params / 1e6:.1f}M")
print(f"   Trainable params: {trainable_params / 1e6:.1f}M")
print(f"   Model size: {total_params * 4 / 1e6:.0f} MB (fp32)")
print(f"   Classification head: {NUM_CLASSES} classes")

## 4. Training Loop

In [ ]:
# === Optimizer & Scheduler ===
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Cosine scheduler with warmup
total_steps = EPOCHS * len(train_loader)
warmup_steps = WARMUP_EPOCHS * len(train_loader)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Loss with label smoothing
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

# Mixed precision scaler
scaler = GradScaler(enabled=FP16)

print(f"✅ Training setup:")
print(f"   Optimizer: AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
print(f"   Scheduler: Cosine with {WARMUP_EPOCHS} warmup epochs")
print(f"   Loss: CrossEntropy + Label Smoothing ({LABEL_SMOOTHING})")
print(f"   Mixed Precision: {FP16}")
print(f"   Total steps: {total_steps}")

In [ ]:
# === Training & Validation functions ===

def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (videos, labels) in enumerate(loader):
        videos = videos.to(device)  # (B, T, C, H, W)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        with autocast(enabled=FP16):
            outputs = model(pixel_values=videos)
            loss = criterion(outputs.logits, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_loss += loss.item()
        _, predicted = outputs.logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        if (batch_idx + 1) % 10 == 0:
            print(f"    Batch {batch_idx+1}/{len(loader)} | "
                  f"Loss: {loss.item():.4f} | "
                  f"Acc: {100.*correct/total:.1f}%", end="\r")
    
    return total_loss / len(loader), 100. * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    for videos, labels in loader:
        videos = videos.to(device)
        labels = labels.to(device)
        
        with autocast(enabled=FP16):
            outputs = model(pixel_values=videos)
            loss = criterion(outputs.logits, labels)
        
        total_loss += loss.item()
        _, predicted = outputs.logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    return total_loss / len(loader), 100. * correct / total, all_preds, all_labels

print("✅ Training functions defined")

In [ ]:
# === TRAINING ===
print("=" * 60)
print(f"🚀 STARTING TRAINING")
print(f"   Model: VideoMAEv2-Small")
print(f"   Classes: {NUM_CLASSES} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE}")
print(f"   Device: {device}")
print("=" * 60)

history = {
    "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": [],
    "lr": []
}
best_val_acc = 0.0
best_epoch = 0

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scheduler, scaler, device
    )
    
    # Validate
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
    
    # Record
    current_lr = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)
    
    elapsed = time.time() - start_time
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        save_path = os.path.join(MODEL_DIR, "videomae_vsl_best")
        model.save_pretrained(save_path)
        # Also save class names
        with open(os.path.join(save_path, "class_names.json"), "w") as f:
            json.dump(class_names, f, ensure_ascii=False)
    
    print(f"  Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train: {train_acc:.1f}% (loss {train_loss:.4f}) | "
          f"Val: {val_acc:.1f}% (loss {val_loss:.4f}) | "
          f"LR: {current_lr:.6f} | "
          f"Time: {elapsed:.0f}s"
          f"{' ⭐ BEST' if val_acc >= best_val_acc else ''}")

print(f"\
{'=' * 60}")
print(f"✅ Training complete!")
print(f"   Best val accuracy: {best_val_acc:.1f}% (epoch {best_epoch})")
print(f"   Model saved to: {MODEL_DIR}/videomae_vsl_best")
print(f"{'=' * 60}")

## 5. Training Curves & Evaluation

In [ ]:
# === Plot training curves ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history["train_loss"], label="Train Loss", color="blue")
axes[0].plot(history["val_loss"], label="Val Loss", color="red")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history["train_acc"], label="Train Acc", color="blue")
axes[1].plot(history["val_acc"], label="Val Acc", color="red")
axes[1].axhline(y=best_val_acc, color="green", linestyle="--", alpha=0.5, label=f"Best: {best_val_acc:.1f}%")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("Training & Validation Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning rate
axes[2].plot(history["lr"], color="green")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Learning Rate")
axes[2].set_title("Learning Rate Schedule (Cosine + Warmup)")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"📊 Training curves saved to {MODEL_DIR}/training_curves.png")

In [ ]:
# === Confusion Matrix & Classification Report ===
print("📊 Final evaluation on validation set...\
")

# Load best model for final evaluation
best_model = VideoMAEForVideoClassification.from_pretrained(
    os.path.join(MODEL_DIR, "videomae_vsl_best"),
    num_labels=NUM_CLASSES,
)
best_model = best_model.to(device)

val_loss, val_acc, all_preds, all_labels = evaluate(
    best_model, val_loader, criterion, device
)

print(f"Best Model Validation Accuracy: {val_acc:.2f}%\
")

# Classification report
print("📋 Classification Report (top 20 classes):")
report = classification_report(
    all_labels, all_preds, 
    target_names=class_names[:NUM_CLASSES],
    output_dict=True
)
print(classification_report(
    all_labels, all_preds,
    target_names=class_names[:NUM_CLASSES],
    zero_division=0
))

# Confusion matrix (show first 20 classes for readability)
max_show = min(20, NUM_CLASSES)
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    cm[:max_show, :max_show], 
    annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names[:max_show],
    yticklabels=class_names[:max_show],
    ax=ax
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix (first {max_show} classes)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Test on Sample Videos

Visualize predictions trên một số video từ validation set.

In [ ]:
import cv2

@torch.no_grad()
def predict_and_visualize(model, video_path: str, class_names: list, num_frames=16):
    """Predict a video and show frames with result."""
    model.eval()
    
    # Load and preprocess
    frames = load_video(video_path, num_frames)
    video = torch.from_numpy(frames).float() / 255.0
    video = video.permute(0, 3, 1, 2)
    
    transformed = []
    for t in range(num_frames):
        frame = Resize(IMAGE_SIZE + 32, antialias=True)(video[t])
        frame = CenterCrop(IMAGE_SIZE)(frame)
        frame = Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])(frame)
        transformed.append(frame)
    
    input_tensor = torch.stack(transformed).unsqueeze(0).to(device)
    
    # Predict
    outputs = model(pixel_values=input_tensor)
    probs = torch.softmax(outputs.logits[0], dim=0)
    top5_probs, top5_idx = torch.topk(probs, 5)
    
    # Visualize
    fig, axes = plt.subplots(2, 8, figsize=(16, 5))
    fig.suptitle(f"True: {Path(video_path).parent.name} | "
                 f"Pred: {class_names[top5_idx[0]]} ({top5_probs[0]:.1%})", fontsize=12)
    
    for i in range(8):
        axes[0, i].imshow(frames[i*2])
        axes[0, i].axis("off")
        axes[0, i].set_title(f"F{i*2}", fontsize=8)
    
    # Top-5 bar chart
    axes[1, 0].remove()
    axes[1, 1].remove()
    axes[1, 2].remove()
    axes[1, 3].remove()
    ax_bar = fig.add_subplot(2, 2, 3)
    colors = ['green' if class_names[idx] == Path(video_path).parent.name else 'steelblue' 
              for idx in top5_idx]
    ax_bar.barh(
        [class_names[idx] for idx in top5_idx.flip(0)],
        top5_probs.flip(0).cpu().numpy(),
        color=colors[::-1]
    )
    ax_bar.set_xlim(0, 1)
    ax_bar.set_title("Top-5 Predictions")
    
    for i in range(4, 8):
        axes[1, i].axis("off")
    
    plt.tight_layout()
    plt.show()

# Show predictions on random val samples
print("🎯 Sample predictions from validation set:\
")
samples = random.sample(val_data, min(5, len(val_data)))
for sample in samples:
    predict_and_visualize(best_model, sample["path"], class_names)
    print()

## 7. Save Training History

Lưu lại history để dùng trong notebook 03.

In [ ]:
# Save training history
history_path = os.path.join(MODEL_DIR, "training_history.json")
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)

print(f"✅ Training history saved to: {history_path}")
print(f"\
📋 Summary:")
print(f"   Best val accuracy: {best_val_acc:.2f}% (epoch {best_epoch})")
print(f"   Model saved at: {MODEL_DIR}/videomae_vsl_best/")
print(f"\
🔜 Next: Run notebook 03_inference_and_deploy.ipynb to:")
print(f"   - Test inference speed")
print(f"   - Export model")
print(f"   - Run Streamlit demo")